# 3. Test

1. 데이터셋을 불러오고, 
2. 네트워크를 정의하며, 
3. 데이터를 불러 descriptor를 추출하고 cluster를 계산해보고. 
4. query와 db사이에서 최근접점을 top-k 알고리즘을 통해 검색한다. 
5. 이때 결과를 UTM을 통해 비교해본다. 


In [1]:
import os
from PIL import Image
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
from collections import namedtuple
import random

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data
import torch.optim as optim
import torch.autograd as Variable


import torchvision
import torchvision.transforms as transforms
import torchvision.models as models


In [3]:

import sklearn
from sklearn.neighbors import NearestNeighbors


In [4]:
import torch.utils.data as data


root_dir = './data/Pittsburgh250k/'
struct_dir = os.path.join(root_dir, 'netvlad_v100_datasets/datasets/')
queries_dir = os.path.join(root_dir, 'queries_real/')


In [5]:
def parse_dbStruct(structfile, dbPath):

    structfile
    dataset = structfile  #db의 이름을 넣기 위한 위치

    mat = loadmat(os.path.join(dbPath,structfile))

    matStruct = mat['dbStruct'].item()

    #debugging 용 출력
    print(len(matStruct))
    first_col = list(map(lambda x: x[0], matStruct))
    for i in range(len(matStruct)):
        print(f"matStruct[{i}] :{first_col[i]}")

    whichSet = matStruct[0].item()

    dbImage = [f[0].item() for f in matStruct[1]]  #이미지리스트
    utmDb = matStruct[2].T

    qImage = [f[0].item() for f in matStruct[3]] #쿼리 이미지
    utmQ = matStruct[4].T

    numDb = matStruct[5].item()
    numQ = matStruct[6].item()

    posDistThr = matStruct[7].item()  #25
    posDistSqThr = matStruct[8].item() #625 --> 25^2
    nonTrivPosDistSqThr = matStruct[9].item() #100 -->10^2

    return dbStruct(whichSet, dataset, dbImage, utmDb, qImage, 
        utmQ, numDb, numQ, posDistThr, 
        posDistSqThr, nonTrivPosDistSqThr)

dbStruct = namedtuple('dbStruct', ['whichSet', 'dataset', 
    'dbImage', 'utmDb', 'qImage', 'utmQ', 'numDb', 'numQ',
    'posDistThr', 'posDistSqThr', 'nonTrivPosDistSqThr'])

test = parse_dbStruct('pitts30k_train.mat',struct_dir)

def input_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
    ])

class WholeDatset(data.Dataset):

    def __init__(self, dbPath, stPath, qPath, structFile, transform=None, onlyDB=False):
        super().__init__()  #parent class 초기화용이나, 현재는 크게필요하지 않음. 
        self.input_transform = transform #tensor로 변환
        self.dbStruct = parse_dbStruct(structFile, stPath) #dataset에 대한 파일 읽기

        self.images = [os.path.join(dbPath, dbIm) for dbIm in self.dbStruct.dbImage]
        if not onlyDB:
            self.images += [os.path.join(qPath, qIm) for qIm in self.dbStruct.qImage]

        self.whichSet = self.dbStruct.whichSet  #train, test, val 중 하나
        self.dataset = self.dbStruct.dataset   # pittsburgh250k, 30k 등

        self.positives = None   #현재는 없음
        self.distances = None   #현재는 없음

    def __len__(self):
            return len(self.images)

    def __getitem__(self, index):
        img = Image.open(self.images[index])  #dataset의 이미지를 불러와 출력

        if self.input_transform:
            img = self.input_transform(img)  #tensor로 변환한다. 
        return img, index

    def getPositive(self):   #학습에선 사용하지 않음. 이후 Test/Evaluation에서 GT추출용으로 사용 
        
        #Data의 숫자가 크지 않아 sklearn으로 아직까지 가능할 듯.         
        if  self.positives is None:
            knn = NearestNeighbors(n_jobs=-1)
            knn.fit(self.dbStruct.utmDb)

            self.distances, self.positives = knn.radius_neighbors(self.dbStruct.utmQ,
                    radius=self.dbStruct.posDistThr)

        return self.positives
    
whole_test_set = WholeDatset(
    dbPath = root_dir,
    stPath = struct_dir,
    qPath = queries_dir,
    structFile= 'pitts30k_test.mat',
    transform=input_transform(),
    onlyDB=False
    )

10
matStruct[0] :train
matStruct[1] :[array(['000/000426_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[585088.30189889 585088.30189889 585088.30189889 ... 584537.10824999
 584537.10824999 584537.10824999]
matStruct[3] :[array(['001/001381_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[585089.36032141 585089.36032141 585089.36032141 ... 584861.3359102
 584861.3359102  584861.3359102 ]
matStruct[5] :[10000]
matStruct[6] :[7416]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]
10
matStruct[0] :test
matStruct[1] :[array(['000/000829_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[2] :[584825.96118815 584825.96118815 584825.96118815 ... 584292.97287724
 584292.97287724 584292.97287724]
matStruct[3] :[array(['000/000546_pitch1_yaw1.jpg'], dtype='<U26')]
matStruct[4] :[584744.96584623 584744.96584623 584744.96584623 ... 584447.89059667
 584447.89059667 584447.89059667]
matStruct[5] :[10000]
matStruct[6] :[6816]
matStruct[7] :[25]
matStruct[8] :[625]
matStruct[9] :[100]


In [6]:
from torch.utils.data import DataLoader, SubsetRandomSampler

def getCluster(mymodel,datasets,K=64):

    #Parameter Initialize
    #테스트용 코드 작성과는 다르게, Ref.코드에서는 이미지 100개에서, 5만개의 Desciptor를 샘플링한다. 
    #따라서, 데이터셋에서 100개의 이미지를 랜덤샘플하고, 여기서 각각 500개의 descriptor를 추출해서 Clustering을 진행한다. 
    #https://m.blog.naver.com/kwangrok21/222412219800 SubsetRandomSampler 사용법법

    from math import ceil
    from torch.utils.data import SubsetRandomSampler
    import numpy as np
    from sklearn.cluster import KMeans
    

    nDesrciptor = 50000
    nPerImages = 100
    nIm = ceil(nDesrciptor / nPerImages)
    sampler = SubsetRandomSampler(np.random.choice(len(datasets), nIm, replace=False))

    data_loader = DataLoader(datasets, sampler=sampler)

    mymodel.eval()
    desc_list = []

    print(data_loader)
    count = 0 
    with torch.no_grad():
        for batch in data_loader:      #data_loader는 DataLoader object이지만, CNN입력은 tensor(B,C,H,W)이어야 하므로, loop로 돌려야 한다. 
            if isinstance(batch, (list, tuple)):
                x = batch[0]          # 보통 image, 지금 이쪽으로 들어온다. 
            else:
                x = batch
            count = count + 1
            
            print(f"x.type is {x.dtype} and {count}, batch shape : {batch[0].shape}")

            x = x.to(device)
            feat = mymodel(x)
            B,C,H,W = feat.shape
            feat = feat.permute(0,2,3,1).reshape(-1,C)

            idx = np.random.choice(feat.shape[0], nPerImages, replace=False)
            feat = feat[idx]

            desc_list.append(feat.cpu().numpy())

    X_np = np.concatenate(desc_list, axis=0)

    print("clustering")
    kmeans = KMeans(n_clusters=K, random_state=0, n_init=10)
    kmeans.fit(X_np)

    centroids = kmeans.cluster_centers_

    return centroids
    
    
    # 이건 getCluster사용하기 전에 먼저 할 것. 
    # x = x.to(device)

In [7]:
class VGG16Feature(nn.Module):
    def __init__(self):
        super().__init__()
        
        #encoder = models.vgg16(pretrained=True) 버전이 바뀌면서 워닝이 뜬다.
        encoder = models.vgg16(weights="VGG16_Weights.IMAGENET1K_FEATURES")
        # capture only feature part and remove last relu and maxpool
        layers = list(encoder.features.children())[:-2]
        
        self.encoder = nn.Sequential(*layers)
        self.encoder_dim = 512
    
    def forward(self, x):
        x = self.encoder(x)
        return x
#Class 끝 

#데이터 준비

transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225]),
])


In [8]:
#cuda 확인
if torch.cuda.is_available() :
    device = 'cuda'
else :
    device = 'cpu' 


In [9]:

model = VGG16Feature().to(device)
centroids = getCluster(model, whole_test_set)    

x.type is torch.float32 and 1, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 2, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 3, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 4, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 5, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 6, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 7, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 8, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 9, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 10, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 11, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 12, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 13, batch shape : torch.Size([1, 3, 480, 640])
x.type is torch.float32 and 14, ba

In [10]:

class NetVLAD(nn.Module):
    #VLAD Layer
    def __init__(self, num_clusters = 64, dim = 128,normalize_input = True):
    
    #        Args:
    # num_clusters : int
    #     The number of clusters
    # dim : int
    #     Dimension of descriptors
    # alpha : float
    #     Parameter of initialization. Larger value is harder assignment.
    # normalize_input : bool
    #     If true, descriptor-wise L2 normalization is applied to input.

        super(NetVLAD, self).__init__()
        self.num_clusters = num_clusters  #cluster개수를 정의해야 vlad vector를 정의할 수 있다. 
        self.dim = dim   #ref코드는 128로 초기화했는데, 만약 VGG-16을 쓴다면 512를 써야 한다. 
        self.alpha = 0  #soft-assignment를 위한 alpha값. scratch 구현에선 2~10정도로 임의 설정했으나, 이제 이걸 학습해나가야 한다. 
        
        self.normalize_input = normalize_input  #이게 필요한가..? 정규화 여부를 저장한다. bool.
        self.conv = nn.Conv2d(dim, num_clusters, kernel_size=(1, 1), bias=False)  #assign용 연산. 아마도 alpha값과 centroid의 벡터곱 등에 쓴다. 
        self.centroids = nn.Parameter(torch.rand(num_clusters, dim))
        #자리 만들기. nn.Parameter 함수를 써서 학습파라미터로 선언한다. none으로 설정하면 네트워크 설정하면서 optimizer를 붙일수 없다. 
        #그래서 뭐라도 넣어놔야 한다. 


    def init_params(self, centroids, descriptors):
        #실제로 cluster의 centroid와 descritor를 받는 부분
        #이 함수를 통해 c_k, alpha, conv.weight와 conv.bias를 통해 assignment score(z_k)를 계산한다. 
        #즉 soft-assignment 관련 항목을 초기화한다. 

        #해야 할것 
        # 1. centroid c_k를 학습할 수 있도록 파라미터로 등록
        # 2. assignment score z_k를 계산하는 self.conv(x)의 weight를 셋팅한다. 
            
            
            #타입 맞추기
            device = descriptors.device
            dtype = descriptors.dtype


            #getCluster를 연산하면 numpy형태로 반환받는다. 그걸 텐서로 바꾼다.
            #기존 scratch에서는 torch.tensor를 사용했고, 이번에는 torch.as_tensor를 사용한다. 
            # 참고자료 https://jh-bk.tistory.com/46
            
            centroids_t = torch.as_tensor(centroids, dtype=dtype, device=device)   # (K=64, dim=512)

            desc_norm = F.normalize(descriptors, p=2, dim=1)       # (M=B*H*W, C)
            cent_norm = F.normalize(centroids_t, p=2, dim=1)       # (K, C)

            # W^T = 2 * alpha * c_k.t() * x_i  

            dots = torch.matmul(cent_norm, desc_norm.t())          
            # (K, M)형태로 출력하기 위해 편의상 transpose의 위치가 바뀐다. 
            # 이는 ref. 코드에서도 동일하다. dots = np.dot(clstsAssign, traindescs.T)
            dots, _ = torch.sort(dots, dim=0, descending=True)

            self.alpha = (-torch.log(torch.tensor(0.01, device=device, dtype=dtype))
                        / torch.mean(dots[0, :] - dots[1, :])).item()

            self.centroids = nn.Parameter(centroids_t)

            self.conv.weight = nn.Parameter(
                (self.alpha * cent_norm).unsqueeze(-1).unsqueeze(-1)   # (K, C, 1, 1)
            )
            self.conv.bias = None

    def forward(self, x):
        N, C = x.shape[:2]

        if self.normalize_input:
            x = F.normalize(x, p=2, dim=1)   # (N, C, H, W)

        soft_assign = self.conv(x)                               # (N, K, H, W)
        soft_assign = soft_assign.view(N, self.num_clusters, -1) # (N, K, HW)
        soft_assign = F.softmax(soft_assign, dim=1)              # (N, K, HW)

        x_flatten = x.view(N, C, -1)                             # (N, C, HW)

        vlad = torch.zeros(
            N, self.num_clusters, C,
            dtype=x.dtype,
            device=x.device
        )                                                        # (N, K, C)

        for k in range(self.num_clusters):
            centroid = self.centroids[k].view(1, C, 1)           # (1, C, 1)
            residual = x_flatten - centroid                      # (N, C, HW)

            assign_weight = soft_assign[:, k, :].view(N, 1, -1)  # (N, 1, HW)
            residual = residual * assign_weight                  # (N, C, HW)

            vlad[:, k, :] = residual.sum(dim=2)                  # (N, C)

        vlad = F.normalize(vlad, p=2, dim=2)                     # (N, K, C)
        vlad = vlad.view(N, -1)                                  # (N, K*C)
        vlad = F.normalize(vlad, p=2, dim=1)                     # (N, K*C)

        return vlad

In [11]:
encoder = VGG16Feature()
pool = NetVLAD(num_clusters=64, dim=512)


model = nn.Module()
model.add_module('encoder', encoder)
model.add_module('pool', pool)

model.to(device)

print(model)

Module(
  (encoder): VGG16Feature(
    (encoder): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace=True)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6): ReLU(inplace=True)
      (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (8): ReLU(inplace=True)
      (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (11): ReLU(inplace=True)
      (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU(inplace=True)
      (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (15): ReLU(inplace=True)
   

In [12]:
x = torch.randn(2,3,224,224).to(device)

feat = model.encoder(x)
print(feat.shape)

vlad = model.pool(feat)
print(vlad.shape)

torch.Size([2, 512, 14, 14])
torch.Size([2, 32768])


In [13]:
loader = DataLoader(whole_test_set, batch_size=4)

images,_ = next(iter(loader))
# images = next(iter(loader))는 에러가 뜬다. 
#아래 코드로 확인해볼것. 
# batch = next(iter(loadbatch = next(iter(loader))
# print(type(batch))
# print(len(batch))
# print(type(batch[0]))

images = images.to(device)

with torch.no_grad():
    desc = model.pool(model.encoder(images))

print(desc.shape)

torch.Size([4, 32768])


In [14]:
all_desc = []
all_indices = []

with torch.no_grad():
    for images, indices in loader:
        images = images.to(device)
        desc = model.pool(model.encoder(images))
        all_desc.append(desc.cpu())
        all_indices.append(indices)

all_desc = torch.cat(all_desc, dim=0)
all_indices = torch.cat(all_indices, dim=0)
print(all_desc.shape)
print(all_indices.shape)

torch.Size([16816, 32768])
torch.Size([16816])


In [15]:
n_db = whole_test_set.dbStruct.numDb
n_q = whole_test_set.dbStruct.numQ

db_desc = all_desc[:n_db]
q_desc = all_desc[n_db:n_db+n_q]

In [16]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

db_desc_np = db_desc.numpy()
q_desc_np = q_desc.numpy()

knn = NearestNeighbors(n_neighbors=10, metric='euclidean')
knn.fit(db_desc_np)

distances, indices = knn.kneighbors(q_desc_np)

print(indices.shape)    # (num_query, 10)
print(distances.shape)  # (num_query, 10)

(6816, 10)
(6816, 10)


In [17]:
from sklearn.neighbors import NearestNeighbors

utmDb = whole_test_set.dbStruct.utmDb
utmQ = whole_test_set.dbStruct.utmQ
posDistThr = whole_test_set.dbStruct.posDistThr

knn_gt = NearestNeighbors(n_jobs=-1)
knn_gt.fit(utmDb)

positives = knn_gt.radius_neighbors(utmQ, radius=posDistThr, return_distance=False)

In [18]:
ks = [1, 5, 10]
recalls = {}

for k in ks:
    correct = 0
    for i in range(len(q_desc_np)):
        pred = indices[i, :k]
        gt = positives[i]
        if np.intersect1d(pred, gt).size > 0:
            correct += 1
    recalls[k] = correct / len(q_desc_np)

print(recalls)

{1: 0.37925469483568075, 5: 0.5759976525821596, 10: 0.667400234741784}
